# 🚗 FocalComm: Hard Instance-Aware Multi-Agent Perception (WACV 2026)
### 📘 Hướng Dẫn Chi Tiết Huấn Luyện & Đánh Giá Mô Hình Trên Kaggle GPU

Notebook này hướng dẫn toàn diện từ A đến Z cách thiết lập môi trường, biên dịch CUDA extensions, tải và xử lý bộ dữ liệu **V2X-Real / DAIR-V2X**, cấu hình tham số, huấn luyện (Training) và đánh giá (Inference) mô hình **FocalComm** trên Kaggle.

---

### ⚙️ Yêu cầu cấu hình Kaggle Notebook:
1. **Accelerator:** GPU (NVIDIA Tesla T4 x 1, T4 x 2 hoặc P100).
2. **Internet:** **ON** (Bắt buộc bật trong cài đặt bên phải để tải packages và dataset).
3. **Persistence:** `Files only` hoặc `Variables and Files`.


## 1. Kiểm tra cấu hình phần cứng & GPU trên Kaggle
Kiểm tra thông số GPU, phiên bản CUDA và trạng thái RAM/Disk.


In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    major, minor = torch.cuda.get_device_capability(0)
    print(f"GPU Compute Capability: sm_{major}{minor}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: CUDA is NOT available. Please switch Accelerator to GPU in Kaggle settings!")

!nvidia-smi


## 2. Clone Repository FocalComm từ GitHub
Clone repo chính thức đã được tối ưu cho môi trường Kaggle.


In [ ]:
# Clone FocalComm repository
!git clone https://github.com/tranvuquanghuy12/FocalComm.git /kaggle/working/FocalComm

# Chuyển thư mục làm việc vào repo
%cd /kaggle/working/FocalComm
!pwd
!ls -la


## 3. Cài đặt các thư viện cần thiết & Biên dịch CUDA Extensions

Mô hình FocalComm sử dụng:
1. `spconv` (Sparse Convolution cho LiDAR Point Cloud 3D)
2. OpenPCDet CUDA Ops (`iou3d_nms_cuda`)
3. Cython extension (`box_overlaps`)
4. Các thư viện xử lý hình học 3D (`open3d`, `pypcd`, `pyquaternion`, `shapely`)


In [ ]:
# 1. Cài đặt các packages cơ bản
!pip install -q easydict numba open3d cython tensorboardX shapely==2.0.0
!pip install -q pyquaternion einops timm icecream python-lzf termcolor efficientnet_pytorch==0.7.0
!pip install -q git+https://github.com/klintan/pypcd.git

# 2. Cài đặt spconv phù hợp với CUDA của Kaggle (CUDA 11.8 hoặc CUDA 12.x)
cuda_version = torch.version.cuda
print(f"Detecting PyTorch CUDA: {cuda_version}")

if cuda_version.startswith("11"):
    !pip install -q spconv-cu118
elif cuda_version.startswith("12"):
    !pip install -q spconv-cu120
else:
    !pip install -q spconv-cu118

# 3. Biên dịch Cython extension cho box_overlaps
%cd /kaggle/working/FocalComm
!python focalcomm/utils/setup.py build_ext --inplace

# 4. Biên dịch CUDA Extension iou3d_nms_cuda (Đã tối ưu auto-detect kiến trúc GPU sm_75 của T4 / sm_60 của P100)
%cd /kaggle/working/FocalComm/focalcomm/pcdet_utils
!python setup.py build_ext --inplace

%cd /kaggle/working/FocalComm
print("\n✅ Hoàn tất cài đặt dependencies và compile C++/CUDA extensions!")


## 4. Tải và Thiết lập Bộ Dữ Liệu V2X-Real

Bộ dữ liệu V2X-Real (LiDAR 128 tia) gồm:
* `val.zip`: 2.23 GB (Dùng để kiểm thử & validation)
* `test.zip`: 5.38 GB (Dùng để benchmark số liệu bài báo)
* `train1.zip` - `train4.zip`: ~19 GB (Tập huấn luyện đầy đủ)

> 💡 **Khuyên dùng trên Kaggle:**
> - Nếu chạy **Dry-Run / Debug / Test:** Tải `val.zip` và `test.zip` là đủ để xác thực toàn bộ pipeline.
> - Nếu huấn luyện toàn bộ: Tải thêm `train1.zip` đến `train4.zip`.


In [ ]:
import os
import urllib.request
import zipfile
import shutil

# Thư mục đích lưu dataset trên Kaggle
DATASET_ROOT = "/kaggle/working/dataset/V2XReal"
os.makedirs(DATASET_ROOT, exist_ok=True)

# Danh sách link Box.com đã giải mã API ID
BOX_FILES = {
    "val.zip": {
        "shared_name": "est8t7lxirg85ohkgoueietxd0xqpf36",
        "file_id": "1668892134513",
        "size_mb": 2279
    },
    "test.zip": {
        "shared_name": "429ak8yk8pawnd84xqx5opcsl0p2mvrp",
        "file_id": "1668588258866",
        "size_mb": 5505
    },
    "train1.zip": {
        "shared_name": "nlnnfbx52m0rjdjwalckcnj3xoc6kckq",
        "file_id": "1668905771343",
        "size_mb": 5701
    },
    "train2.zip": {
        "shared_name": "98rgies3hn0uw6d5oecajsru99lmvi0q",
        "file_id": "1669045514219",
        "size_mb": 5107
    },
    "train3.zip": {
        "shared_name": "d1fuuvx4hisfxkxi1h0gipxfn7tpaony",
        "file_id": "1669051618933",
        "size_mb": 4912
    },
    "train4.zip": {
        "shared_name": "tre8f2n816n4dfqodp74cnqfz7on5qw2",
        "file_id": "1668880852107",
        "size_mb": 3470
    }
}

def download_and_extract_box_file(fname, info, target_dir):
    zip_path = os.path.join(target_dir, fname)
    download_url = f"https://ucla.app.box.com/index.php?rm=box_download_shared_file&shared_name={info['shared_name']}&file_id=f_{info['file_id']}"
    
    print(f"\n⬇️ Đang tải {fname} (~{info['size_mb']} MB)...")
    req = urllib.request.Request(download_url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
    with urllib.request.urlopen(req) as resp, open(zip_path, 'wb') as out_file:
        shutil.copyfileobj(resp, out_file)
    print(f"✅ Đã tải xong {fname}. Đang giải nén...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print(f"📦 Đã giải nén xong {fname}.")

    # Xóa file zip để bảo toàn dung lượng đĩa
    os.remove(zip_path)
    print(f"🗑️ Đã dọn file nén {fname} để tiết kiệm dung lượng đĩa.")

# Chọn các phần cần tải (Mặc định tải val và test để kiểm thử trước)
# Nếu muốn train full, thêm: "train1.zip", "train2.zip", "train3.zip", "train4.zip"
DOWNLOAD_TARGETS = ["val.zip", "test.zip"]

for target in DOWNLOAD_TARGETS:
    download_and_extract_box_file(target, BOX_FILES[target], DATASET_ROOT)

print("\n📂 Cấu trúc thư mục dataset sau khi tải:")
!ls -lh /kaggle/working/dataset/V2XReal


## 5. Tải Pretrained Checkpoints Chính Thức từ Hugging Face
Nếu muốn kiểm thử (Inference) hoặc Fine-tuning từ trọng số chuẩn của bài báo (WACV 2026), tải checkpoint chính thức:


In [ ]:
!pip install -q huggingface_hub

# Tải checkpoint chính thức từ Hugging Face scdrand23/FocalComm
!python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='scdrand23/FocalComm', local_dir='checkpoints')"

!ls -lh checkpoints/
!ls -lh checkpoints/focalcomm_v2xreal/


## 6. Cập nhật file cấu hình YAML (`focalcommv3.yaml`)
Cấu hình lại đường dẫn dữ liệu, batch size và chế độ huấn luyện cho tương thích với tài nguyên Kaggle.


In [ ]:
import yaml

config_path = "/kaggle/working/FocalComm/focalcomm/hypes_yaml/v2xreal/focalcommv3.yaml"

with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Cập nhật đường dẫn dataset trên Kaggle
cfg['root_dir'] = "/kaggle/working/dataset/V2XReal/train"
cfg['validate_dir'] = "/kaggle/working/dataset/V2XReal/val"
cfg['test_dir'] = "/kaggle/working/dataset/V2XReal/test"

# Điều chỉnh batch size và num_workers để tránh OOM trên Kaggle GPU (16GB VRAM)
cfg['train_params']['batch_size'] = 2  # Phù hợp với GPU T4 16GB
cfg['train_params']['epoches'] = 30
cfg['dataset_mode'] = 'vc'             # 'vc': Vehicle-Centric, 'ic': Infrastructure-Centric

# Lưu lại file cấu hình mới
kaggle_cfg_path = "/kaggle/working/FocalComm/focalcomm/hypes_yaml/v2xreal/focalcommv3_kaggle.yaml"
with open(kaggle_cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"✅ Đã tạo file cấu hình cho Kaggle tại: {kaggle_cfg_path}")


## 7. Khởi Chạy Huấn Luyện (Training)

Có 2 chế độ:
1. **Chế độ Debug / Dry-Run (`--subset`):** Huấn luyện trên 4 kịch bản mẫu để kiểm tra toàn bộ luồng mất mát (Loss) và gradient xem có lỗi OOM hay lỗi shape không.
2. **Chế độ Huấn luyện Đầy đủ:** Bỏ cờ `--subset`.


In [ ]:
# Chạy Dry-run kiểm tra code trước (1-2 epochs)
!python focalcomm/tools/train_focalcomm.py \
    --hypes_yaml focalcomm/hypes_yaml/v2xreal/focalcommv3_kaggle.yaml \
    --subset


## 8. Đánh Giá Mô Hình (Inference & Evaluation)
Chạy script `inference.py` để tính toán độ chính xác:
* **AP@0.3** và **AP@0.5** cho từng lớp: **Car**, **Pedestrian**, **Truck**.
* **Overall mAP**.


In [ ]:
# Chạy đánh giá trên tập Test sử dụng checkpoint chính thức
!python focalcomm/tools/inference.py \
    --model_dir checkpoints/focalcomm_v2xreal \
    --fusion_method intermediate


## 9. Lưu Trữ Checkpoints & Kết Quả Thực Nghiệm
Nén các checkpoint tốt nhất (`net_epoch_best.pth`) và file log để tải về máy hoặc lưu vào Google Drive / Kaggle Output.


In [ ]:
import shutil

# Tạo thư mục lưu trữ kết quả cuối cùng
output_archive = "/kaggle/working/focalcomm_results.tar.gz"
!tar -czvf /kaggle/working/focalcomm_best_models.tar.gz -C /kaggle/working/FocalComm/focalcomm/tools/ logs/ 2>/dev/null || true

print("✅ Đã đóng gói các logs và checkpoint vào /kaggle/working/")
!ls -lh /kaggle/working/*.tar.gz
